# Pre-process Steam

Same pipeline as MovieLens's `process.ipynb` (dedup → k-core → map ids → leave-one-out split → product table → simulator jsonl), adapted for the Steam schema: reviews/games loaded straight from the loose-JSON `.json.gz` dumps (see `explore_raw.ipynb`) instead of a header-less `.dat`, implicit feedback (playtime-based reviews with no rating field, so no rating-threshold filter), and `genres`/`tags` as native JSON lists instead of MovieLens's pipe-delimited string.

# 0. Import & logging

In [1]:
import os
import re
import ast
import gzip
import json
import pickle
import logging
import itertools
from local_package.config.data import STEAM_RAW_DIR, STEAM_PROCESSED_DIR
from local_package.config.log import setup_logging, STEAM_PROCESS_LOG_DIR
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
logger = setup_logging(name="process", level=logging.INFO, to_file=True, log_dir=STEAM_PROCESS_LOG_DIR)

# 1. Configuration

In [3]:
# Seed
SEED = 2024

In [4]:
# Path
DATA_DIR = STEAM_RAW_DIR  # matches explore_raw.ipynb

REVIEWS_FILE = DATA_DIR / "steam_reviews.json.gz"
GAMES_FILE = DATA_DIR / "steam_games.json.gz"

# Reviews are undocumented (see explore_raw.ipynb); these are the columns confirmed to be
# reliably populated there, not the whole candidate list explore_raw checks
REVIEW_USER_COL = "username"
REVIEW_ITEM_COL = "product_id"
REVIEW_TIME_COL = "date"

OUTPUT_DIR = STEAM_PROCESSED_DIR / "chatbot"  # where train/valid/test/products/simulator files go
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# Min interactions per user/item for k-core filtering
USER_K = 5  # k-core: min interactions per user
ITEM_K = 5  # k-core: min interactions per item

# Maximum lengths for simulator strings
MAX_HISTORY_LEN = 10  # max past items shown in a simulator history string
MAX_TITLE_LEN = 50  # max chars of a game title used in simulator strings

# Simulator sampling
SIMULATOR_SAMPLE_N = 900  # test users sampled for the simulator jsonl export

# 2. Load raw data

In [6]:
def iter_loose_json_gz(path):
    """Yield one parsed record per line. These dumps are 'loose' JSON (Python dict literals,
    single-quoted) rather than strict JSON, so plain json.loads fails on them. Same helper as
    explore_raw.ipynb, kept here instead of imported so this notebook has no dependency on it."""
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            yield ast.literal_eval(line)

In [7]:
def load_reviews_and_games(reviews_file, games_file):
    logger.info("Loading reviews from %s", reviews_file)
    review_df = pd.DataFrame.from_records(iter_loose_json_gz(reviews_file))

    logger.info("Loading games from %s", games_file)
    game_df = pd.DataFrame.from_records(iter_loose_json_gz(games_file))
    game_df = game_df.drop(columns=["title"])
    game_df = game_df.rename(columns={"app_name": "title"})
    
    logger.info("Shape of reviews: %s", review_df.shape)
    logger.info("Shape of games: %s", game_df.shape)
    return review_df, game_df

In [8]:
review_df, game_df = load_reviews_and_games(REVIEWS_FILE, GAMES_FILE)

2026-09-07 08:53:38 [INFO] process: Loading reviews from C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\raw\steam\steam_reviews.json.gz
2026-09-07 09:07:43 [INFO] process: Loading games from C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\raw\steam\steam_games.json.gz
2026-09-07 09:07:48 [INFO] process: Shape of reviews: (7793069, 12)
2026-09-07 09:07:48 [INFO] process: Shape of games: (32135, 15)


# 3. Clean metadata

In [9]:
def remove_html_tags(text):
    return BeautifulSoup(text, "html.parser").get_text()


def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub("", text)


def remove_special_characters(text, remove_digits=False):
    pattern = r"[^a-zA-Z0-9\s\u4e00-\u9fa5,.!]" if not remove_digits else r"[^a-zA-Z\s\u4e00-\u9fa5,.!]"
    return re.sub(pattern, "", text)


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = remove_html_tags(text)
    text = remove_emojis(text)
    return remove_special_characters(text)

In [10]:
def parse_price(x):
    # prices come as strings like "$29.99" or "Free"/"Free to Play"; a handful are already
    # floats (NaN) if the field was missing entirely
    if isinstance(x, str):
        match = re.findall(r"\d+\.?\d*", x)
        return float(match[0]) if match else 0.0
    return x if isinstance(x, float) else 0.0

In [11]:
def genres_to_category_and_description(genres):
    # genres/tags arrive as native Python lists (parsed straight out of the loose JSON), unlike
    # MovieLens's pipe-delimited string -- analogous to join_genres in the MovieLens pipeline
    if not isinstance(genres, list) or not genres:
        return "Unknown", "No description"
    return genres[0], ", ".join(genres)

In [14]:
game_df = game_df[~game_df["id"].isna()]
game_df = game_df[~game_df["title"].isna()].reset_index(drop=True)
logger.info("Games after dropping missing id/app_name: %s", game_df.shape)

2026-09-07 09:09:04 [INFO] process: Games after dropping missing id/app_name: (32132, 15)


In [16]:
game_df["id"] = game_df["id"].astype(int)
game_df["price"] = game_df["price"].apply(parse_price)
game_df["price"] = game_df["price"].fillna(game_df["price"].mean())

game_df["title"] = game_df["title"].apply(clean_text)
game_df["release_date"] = pd.to_datetime(game_df["release_date"], format="%b %d, %Y", errors="coerce")
game_df[["category", "description"]] = game_df["genres"].apply(
    lambda g: pd.Series(genres_to_category_and_description(g))
)

game_df = game_df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)
game_df = game_df.rename(columns={"app_name": "title"})

# 4. Filter interactions

In [17]:
def get_valid_ids(df, col_name, k):
    frequency = df.groupby([col_name])[[col_name]].count()
    return frequency[frequency[col_name] >= k].index

In [18]:
def keep_first_filter(df, user_col=REVIEW_USER_COL, item_col=REVIEW_ITEM_COL, time_col=REVIEW_TIME_COL):
    logger.info("Keeping first interaction per duplicated review, begin: %s", df.shape)
    df = df.sort_values(by=[user_col, time_col]).reset_index(drop=True)
    df = df.drop_duplicates(subset=[user_col, item_col], keep="first").reset_index(drop=True)
    logger.info("After keep-first filter: %s", df.shape)
    return df

In [19]:
def k_core_filter(df, user_k=USER_K, item_k=ITEM_K, user_col=REVIEW_USER_COL, item_col=REVIEW_ITEM_COL, max_iter=20):
    logger.info("k-core filtering (user_k=%d, item_k=%d), begin: %s", user_k, item_k, df.shape)
    num_users_prev, num_items_prev = len(df[user_col].unique()), len(df[item_col].unique())
    delta, it = True, 0
    while delta and it < max_iter:
        valid_users = get_valid_ids(df, user_col, user_k)
        df = df[df[user_col].isin(valid_users)]
        valid_items = get_valid_ids(df, item_col, item_k)
        df = df[df[item_col].isin(valid_items)]
        num_users, num_items = len(valid_users), len(valid_items)
        delta = (num_users != num_users_prev) or (num_items != num_items_prev)
        logger.info("Iter %d: users %d/%d, items %d/%d", it, num_users, num_users_prev, num_items, num_items_prev)
        num_users_prev, num_items_prev = num_users, num_items
        it += 1
    logger.info("After k-core filter: %s", df.shape)
    return df

In [20]:
review_df_0 = review_df[[REVIEW_USER_COL, REVIEW_ITEM_COL, REVIEW_TIME_COL]]
review_df_0 = review_df_0[~review_df_0[REVIEW_USER_COL].isna()].reset_index(drop=True)

# product_id comes out of the reviews JSON as a string while games' id is an int -- align
# dtypes before the isin, or every row silently fails to match and this filters everything out
review_df_0[REVIEW_ITEM_COL] = pd.to_numeric(review_df_0[REVIEW_ITEM_COL], errors="coerce").astype("Int64")
review_df_0 = review_df_0[~review_df_0[REVIEW_ITEM_COL].isna()].reset_index(drop=True)
review_df_0[REVIEW_ITEM_COL] = review_df_0[REVIEW_ITEM_COL].astype(int)
review_df_0 = review_df_0[review_df_0[REVIEW_ITEM_COL].isin(game_df["id"])].reset_index(drop=True)

data_df = keep_first_filter(review_df_0)
data_df = k_core_filter(data_df).reset_index(drop=True)
data_df = data_df.rename(columns={REVIEW_USER_COL: "user_id", REVIEW_ITEM_COL: "item_id"})

2026-09-07 09:09:35 [INFO] process: Keeping first interaction per duplicated review, begin: (7793069, 3)
2026-09-07 09:09:52 [INFO] process: After keep-first filter: (6889728, 3)
2026-09-07 09:09:52 [INFO] process: k-core filtering (user_k=5, item_k=5), begin: (6889728, 3)
2026-09-07 09:10:00 [INFO] process: Iter 0: users 281645/2567538, items 11978/15474
2026-09-07 09:10:05 [INFO] process: Iter 1: users 281214/281645, items 11961/11978
2026-09-07 09:10:09 [INFO] process: Iter 2: users 281210/281214, items 11961/11961
2026-09-07 09:10:14 [INFO] process: Iter 3: users 281210/281210, items 11961/11961
2026-09-07 09:10:14 [INFO] process: After k-core filter: (3484694, 3)


# 5. Mapping users and items' IDs

In [21]:
def map_id(df, user_colname="user_id", item_colname="item_id"):
    logger.info("Mapping user and item ids to contiguous integers")
    users, items = df[user_colname].unique(), df[item_colname].unique()
    user_map = {u: k + 1 for k, u in enumerate(users)}
    item_map = {i: k + 1 for k, i in enumerate(items)}
    df[user_colname] = df[user_colname].apply(lambda x: user_map[x])
    df[item_colname] = df[item_colname].apply(lambda x: item_map[x])
    return df, user_map, item_map

In [22]:
data_df, user_map, item_map = map_id(data_df)

2026-09-07 09:10:14 [INFO] process: Mapping user and item ids to contiguous integers


In [23]:
# Save the id maps to a json file for later use
# (json keys must be strings; Steam ids are ints, so cast them going in)
with open(os.path.join(OUTPUT_DIR, "map.json"), "w") as f:
    json.dump(
        {"item": {str(k): v for k, v in item_map.items()}, "user": {str(k): v for k, v in user_map.items()}}, f
    )
logger.info("Saved id maps to %s", os.path.join(OUTPUT_DIR, "map.json"))

2026-09-07 09:10:19 [INFO] process: Saved id maps to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\steam\chatbot\map.json


# 6. Leave-one-out split

In [24]:
def split_leave_one_out_seq(data, col_name, time_colname, col_names_2_return):
    df_sorted = data.sort_values(by=[col_name, time_colname]).reset_index(drop=True)
    df_test = df_sorted.groupby(by=col_name, as_index=False).nth(-1)
    df_train = df_sorted.iloc[df_sorted.index.difference(df_test.index)]
    return (
        df_train.reset_index(drop=True)[col_names_2_return],
        df_test.reset_index(drop=True)[col_names_2_return],
    )

In [25]:
df_train_0, df_test = split_leave_one_out_seq(data_df, "user_id", "date", ["user_id", "item_id", "date"])
df_train, df_valid = split_leave_one_out_seq(df_train_0, "user_id", "date", ["user_id", "item_id"])

In [26]:
df_train.to_csv(os.path.join(OUTPUT_DIR, "train.tsv"), index=None)
df_valid.to_csv(os.path.join(OUTPUT_DIR, "valid.tsv"), index=None)
df_test.to_csv(os.path.join(OUTPUT_DIR, "test.tsv"), index=None)
df_train_0.to_csv(os.path.join(OUTPUT_DIR, "user_history.tsv"), index=None)

logger.info(
    "Saved splits to %s (train=%d, valid=%d, test=%d, full_history=%d)",
    OUTPUT_DIR, len(df_train), len(df_valid), len(df_test), len(df_train_0),
)

2026-09-07 09:10:31 [INFO] process: Saved splits to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\steam\chatbot (train=2922274, valid=281210, test=281210, full_history=3203484)


# 7. Product table

Named `products.*` (not `games.*`) to match the MovieLens/Amazon output layout, so downstream chatbot/recsys code can read any domain's processed folder the same way.

In [27]:
saved_meta_df = game_df[game_df["id"].isin(item_map.keys())]
saved_meta_df = saved_meta_df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)
saved_meta_df["id"] = saved_meta_df["id"].apply(lambda x: item_map[x])

In [ ]:
user_history = df_train_0.groupby("user_id").agg(list)
item_count = user_history["item_id"].explode().value_counts()
saved_meta_df["visited_num"] = saved_meta_df["id"].apply(lambda x: item_count.loc[x] if x in item_count else 0)

In [30]:
saved_meta_df["metascore"] = pd.to_numeric(saved_meta_df["metascore"], errors="coerce")
saved_meta_df.to_feather(os.path.join(OUTPUT_DIR, "products.ftr"))

saved_meta_df.to_csv(os.path.join(OUTPUT_DIR, "products.csv"), index=None, sep="|")
logger.info("Saved product table (%d items) to %s", len(saved_meta_df), OUTPUT_DIR)

2026-09-07 09:11:54 [INFO] process: Saved product table (11961 items) to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\steam\chatbot


# 8. Simulator jsonl sample

In [31]:
def write_jsonl(obj, fpath):
    try:
        with open(fpath, "w") as outfile:
            for entry in obj:
                json.dump(entry, outfile)
                outfile.write("\n")
        logger.info("Saved %d records to %s", len(obj), fpath)
    except Exception as e:
        fallback = f"{fpath}.pkl"
        logger.exception("Failed to write jsonl (%s), falling back to pickle at %s", e, fallback)
        with open(fallback, "wb") as tempfile:
            pickle.dump(obj, tempfile)

In [32]:
saved_meta_df_indexed = saved_meta_df.set_index("id")
id2title = {id_: saved_meta_df_indexed.loc[id_].title[:MAX_TITLE_LEN] for id_ in saved_meta_df_indexed.index}

In [33]:
n_sample = min(SIMULATOR_SAMPLE_N, len(df_test))
test_data = df_test.sample(n_sample, random_state=SEED)
test_data["history"] = test_data["user_id"].apply(
    lambda x: "; ".join([id2title[i] for i in user_history.loc[x]["item_id"][-MAX_HISTORY_LEN:]])
)
test_data["target"] = test_data["item_id"].apply(lambda x: saved_meta_df_indexed.loc[x].title)
test_data.reset_index(drop=True, inplace=True)

In [34]:
simulator_path = os.path.join(OUTPUT_DIR, f"simulator_test_data_{n_sample}.jsonl")
write_jsonl(test_data[["history", "target"]].to_dict("records"), simulator_path)
logger.info("Pipeline complete.")

2026-09-07 09:12:00 [INFO] process: Saved 900 records to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\steam\chatbot\simulator_test_data_900.jsonl
2026-09-07 09:12:00 [INFO] process: Pipeline complete.
